# Notebook 2 : 02_Landing_to_Bronze

#### Objective

###### The objective of this notebook is to read the raw CSV file from the Landing folder and load it into the Bronze layer as a Delta table.

###### The Bronze layer preserves the original data exactly as received from the source. No cleaning or validation is performed in this layer. Only ingestion metadata is added to track the data loading process.

## STEP 1: Import Libraries

In [0]:
from pyspark.sql.functions import * 
from pyspark.sql.types import *
from datetime import *

## STEP 2: Configuration

In [0]:
landing_path= "/Volumes/dbacademy/default/myvolume/Project Sentinel"
bronze_path = "/Volumes/dbacademy/default/myvolume/bronze"
bronze_table = "bronze_upi_transactions"

## Step 3 – Read Landing CSV

In [0]:
landing_df = spark.read.format("csv") \
    .option("header",True)\
    .option("inferSchema", True) \
    .load(landing_path)

display(landing_df.limit(10))

transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason
2bbc7344-a5dc-4ffb-bff0-171c1c7620c0,2026-07-08T14:04:52.000Z,HDFC,SBI,ankit557@okhdfc,rohit684@oksbi,7090,P2P,SUCCESS,Indore,Madhya Pradesh,DEV100111,2366,null
61f432e0-bd55-4cfe-b14d-2beec8fd638f,2026-06-15T06:45:32.000Z,ICICI,HDFC,priya403@icici,manish738@okhdfc,15597,P2P,SUCCESS,Mumbai,Maharashtra,DEV100174,1016,null
bb370089-0c45-4264-bc56-15f741cbef38,2026-06-25T21:59:21.000Z,ICICI,SBI,manish200@icici,ankit279@oksbi,40824,Merchant,SUCCESS,Jaipur,Rajasthan,DEV100122,217,null
62fb68f6-15e6-4ee9-9478-f772f83d9e33,2026-06-18T15:32:26.000Z,ICICI,HDFC,neha849@icici,arjun756@okhdfc,18156,Recharge,SUCCESS,Hyderabad,Telangana,DEV100003,473,null
79b0fb85-a310-48a2-957b-92e38f6e789f,2026-07-06T08:11:01.000Z,HDFC,Axis,rohit687@okhdfc,arjun923@axis,24257,P2P,SUCCESS,Pune,Maharashtra,DEV100140,2425,null
e0edde04-9c30-43aa-8538-c800a9a3c1ff,2026-06-30T21:33:28.000Z,Axis,HDFC,arjun816@axis,pooja764@okhdfc,33603,P2P,SUCCESS,Bangalore,Karnataka,DEV100014,1272,null
c0de5084-2cb8-412b-976e-79202956f294,2026-07-06T02:01:28.000Z,SBI,Axis,kartik629@oksbi,divya195@axis,14605,Bill Payment,SUCCESS,Pune,Maharashtra,DEV100034,1279,null
3069fc99-7521-402d-8ca6-a10ecdb5cd87,2026-07-04T06:39:40.000Z,HDFC,Axis,pooja190@okhdfc,priya798@axis,17898,Recharge,SUCCESS,Indore,Madhya Pradesh,DEV100174,1996,null
95276c83-5a4c-47b5-96f8-02fa3897504c,2026-06-14T14:36:28.000Z,Axis,SBI,karan234@axis,kartik537@oksbi,21863,P2P,SUCCESS,Hyderabad,Telangana,DEV100193,989,null
eefed37e-8689-43d8-a13a-bbca7b1e841f,2026-06-17T15:31:51.000Z,Axis,ICICI,riya328@axis,prachi153@icici,null,Recharge,SUCCESS,Mumbai,Maharashtra,DEV100088,469,null


#### 3.1 : Check Schema

In [0]:
landing_df.printSchema()

root
 |-- transaction_id: string (nullable = true)
 |-- transaction_timestamp: timestamp (nullable = true)
 |-- sender_bank: string (nullable = true)
 |-- receiver_bank: string (nullable = true)
 |-- sender_upi: string (nullable = true)
 |-- receiver_upi: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- transaction_type: string (nullable = true)
 |-- transaction_status: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- device_id: string (nullable = true)
 |-- response_time_ms: integer (nullable = true)
 |-- failure_reason: string (nullable = true)



#### 3.2 : Check Record Count

In [0]:
print("Total Records :", landing_df.count())

Total Records : 20400


## Step 4 : Add Metadata

### 4.1 Add Load Timestamp

In [0]:
bronze_df = landing_df.withColumn("load_timestamp",current_timestamp())

### 4.2 Add Source File

In [0]:
bronze_df = bronze_df.withColumn("source_file",col("_metadata.file_path"))

In [0]:
display(bronze_df.limit(5))

transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason,load_timestamp,source_file
2bbc7344-a5dc-4ffb-bff0-171c1c7620c0,2026-07-08T14:04:52.000Z,HDFC,SBI,ankit557@okhdfc,rohit684@oksbi,7090,P2P,SUCCESS,Indore,Madhya Pradesh,DEV100111,2366,null,2026-07-12T18:25:25.321Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
61f432e0-bd55-4cfe-b14d-2beec8fd638f,2026-06-15T06:45:32.000Z,ICICI,HDFC,priya403@icici,manish738@okhdfc,15597,P2P,SUCCESS,Mumbai,Maharashtra,DEV100174,1016,null,2026-07-12T18:25:25.321Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
bb370089-0c45-4264-bc56-15f741cbef38,2026-06-25T21:59:21.000Z,ICICI,SBI,manish200@icici,ankit279@oksbi,40824,Merchant,SUCCESS,Jaipur,Rajasthan,DEV100122,217,null,2026-07-12T18:25:25.321Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
62fb68f6-15e6-4ee9-9478-f772f83d9e33,2026-06-18T15:32:26.000Z,ICICI,HDFC,neha849@icici,arjun756@okhdfc,18156,Recharge,SUCCESS,Hyderabad,Telangana,DEV100003,473,null,2026-07-12T18:25:25.321Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
79b0fb85-a310-48a2-957b-92e38f6e789f,2026-07-06T08:11:01.000Z,HDFC,Axis,rohit687@okhdfc,arjun923@axis,24257,P2P,SUCCESS,Pune,Maharashtra,DEV100140,2425,null,2026-07-12T18:25:25.321Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv


## Step 5 : Write Bronze Delta Table

In [0]:
bronze_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(bronze_table)

print("Bronze Table Created Successfully")

Bronze Table Created Successfully


## Step 6 : Verify Bronze Layer

In [0]:
b_table = spark.table(bronze_table)
display(b_table.limit(5))

transaction_id,transaction_timestamp,sender_bank,receiver_bank,sender_upi,receiver_upi,amount,transaction_type,transaction_status,city,state,device_id,response_time_ms,failure_reason,load_timestamp,source_file
2bbc7344-a5dc-4ffb-bff0-171c1c7620c0,2026-07-08T14:04:52.000Z,HDFC,SBI,ankit557@okhdfc,rohit684@oksbi,7090,P2P,SUCCESS,Indore,Madhya Pradesh,DEV100111,2366,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
61f432e0-bd55-4cfe-b14d-2beec8fd638f,2026-06-15T06:45:32.000Z,ICICI,HDFC,priya403@icici,manish738@okhdfc,15597,P2P,SUCCESS,Mumbai,Maharashtra,DEV100174,1016,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
bb370089-0c45-4264-bc56-15f741cbef38,2026-06-25T21:59:21.000Z,ICICI,SBI,manish200@icici,ankit279@oksbi,40824,Merchant,SUCCESS,Jaipur,Rajasthan,DEV100122,217,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
62fb68f6-15e6-4ee9-9478-f772f83d9e33,2026-06-18T15:32:26.000Z,ICICI,HDFC,neha849@icici,arjun756@okhdfc,18156,Recharge,SUCCESS,Hyderabad,Telangana,DEV100003,473,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
79b0fb85-a310-48a2-957b-92e38f6e789f,2026-07-06T08:11:01.000Z,HDFC,Axis,rohit687@okhdfc,arjun923@axis,24257,P2P,SUCCESS,Pune,Maharashtra,DEV100140,2425,null,2026-07-12T18:25:28.953Z,dbfs:/Volumes/dbacademy/default/myvolume/Project%20Sentinel/upi_transactions.csv
